# Bacpipe Tutorial

This notebook walks through the main workflows of the `bacpipe` library — from inspecting configuration and loading audio, through generating embeddings with built-in and custom models, to running the full pipeline and benchmarking classifier performance.



Make sure to reset your config and settings files before running this notebook, 
as they may contain settings from previous runs that could prevent this notebook 
from finding the correct paths.
---

**Contents**
1. [Setup & Configuration](#1.-Setup-&-Configuration)
2. [Loading Audio Files](#2.-Loading-Audio-Files)
3. [Ground Truth Labels](#3.-Ground-Truth-Labels)
4. [End-to-End: bacpipe.play](#4.-End-to-End:-bacpipe.play)
5. [Full Pipeline - Single Model](#5.-Full-Pipeline---Single-Model)
6. [Full Pipeline - Multiple Models](#6.-Full-Pipeline---Multiple-Models)
7. [Generating Embeddings - Full Directory (Save to Disk)](#7.-Generating-Embeddings---Full-Directory-(Save-to-Disk))
8. [Generating Embeddings - Single File (In-Memory)](#8.-Generating-Embeddings---Single-File-(In-Memory))
9. [High-Level Workflow - generate_embeddings](#9.-High-Level-Workflow---generate_embeddings)
10. [Benchmarking Classifier Performance](#10.-Benchmarking-Classifier-Performance)

---
## 0. Imports & Working Directory

Set the working directory to the repo root and import the display utility.

In [1]:
# to run successfully the packages for jupyter notebook need to be installed:
# uv pip install ipykernel, ipython

from IPython.display import display
import os
from pathlib import Path

# load the specific package
import bacpipe

In [2]:
# get the path to the folder where the notebook is located
notebook_path = Path().resolve()

# get the name of the current notebook file
notebook_file = 'simple_use_cases'

# get the directoy where the notebook is located
notebook_dir = notebook_path.name

# working directory 
wd_path = notebook_path.parent

# Change the value of the key main_results_dir in the namespace bacpipe.settings to change the directory 
# where the results of the tutorials are stored. By default, it is set to './bacpipe_results'.
bacpipe.settings.main_results_dir = str(wd_path / 'bacpipe_results')

# !WARNING! the following code deletes the folder where the results of this tutorial is stored to be sure to start with a clean folder. 
# If you have important data in this folder, please comment it before running this code.
folder_path = bacpipe.settings.main_results_dir
if os.path.exists(folder_path):
    # Prompt the user
    user_input = input(f"Are you sure you want to delete '{folder_path}'? (y/n): ").lower().strip()

    if user_input == 'y':
        import shutil
        shutil.rmtree(folder_path) 
        print(f"Folder {folder_path} deleted.")
    else:
        print("Operation cancelled.")

else:
    print(f"Folder {folder_path} not found.")

Folder /media/haupert/data/mes_projets/_cesco-ia/session_20260908/tp_bacpipe/bacpipe_results deleted.


---
## 1. Setup & Configuration

Import `bacpipe` and inspect the current configuration and settings. You can also list all available API endpoints, supported models, and the embedding dimensions for each model.

In [3]:
import bacpipe

print('Config:')
display(bacpipe.config)

print('Settings:')
display(bacpipe.settings)

Config:


namespace(audio_dir='bacpipe/tests/test_data',
          overwrite=False,
          dashboard=True,
          models=['birdnet', 'perch_bird'],
          already_computed=False,
          dim_reduction_model='umap',
          evaluation_task=[])

Settings:


namespace(main_results_dir='/media/haupert/data/mes_projets/_cesco-ia/session_20260908/tp_bacpipe/bacpipe_results',
          embed_parent_dir='embeddings',
          dim_reduc_parent_dir='dim_reduced_embeddings',
          evaluations_dir='evaluations',
          model_base_path='bacpipe_model_checkpoints',
          device='cpu',
          global_batch_size=8,
          audio_suffixes=['.wav',
                          '.WAV',
                          '.aif',
                          '.mp3',
                          '.MP3',
                          '.flac',
                          '.ogg'],
          padding='wrap',
          avoid_pipelined_gpu_inference=False,
          nr_parallel_workers=False,
          rm_embedding_on_keyboard_interrupt=False,
          check_if_already_processed=True,
          check_if_already_dim_reduced=True,
          label_column='species',
          annotations_filename='annotations.csv',
          only_embed_annotations=False,
          min_annotat

---
## 2. Loading Audio Files

Retrieve all audio files from a directory as a list of strings.

`bacpipe.get_audio_files` recursively finds every audio file under the given
directory and returns the paths as a list. By default the paths are returned as
`pathlib.Path` objects; pass `return_type='str'` to get plain strings instead.
The supported file extensions are controlled by `bacpipe.settings.audio_suffixes`
(e.g. `.wav`, `.mp3`, `.flac`).


In [4]:
# audio files directory
bacpipe.config.audio_dir = str( wd_path / 'bacpipe_data') 

audio_files = bacpipe.get_audio_files(
    bacpipe.config.audio_dir , return_type='str'
)
audio_files

finding audio files: 157it [00:00, 184094.42it/s]
Found 150 number of audio files.


['/media/haupert/data/mes_projets/_cesco-ia/session_20260908/tp_bacpipe/bacpipe_data/ambient_sound/ambient_sound01.wav',
 '/media/haupert/data/mes_projets/_cesco-ia/session_20260908/tp_bacpipe/bacpipe_data/ambient_sound/ambient_sound02.wav',
 '/media/haupert/data/mes_projets/_cesco-ia/session_20260908/tp_bacpipe/bacpipe_data/erirub/MNHN-SO-2020-421_extr3_0s_00_norm.wav',
 '/media/haupert/data/mes_projets/_cesco-ia/session_20260908/tp_bacpipe/bacpipe_data/erirub/MNHN-SO-2020-421_extr3_0s_01_norm.wav',
 '/media/haupert/data/mes_projets/_cesco-ia/session_20260908/tp_bacpipe/bacpipe_data/erirub/MNHN-SO-2020-421_extr3_0s_02_norm.wav',
 '/media/haupert/data/mes_projets/_cesco-ia/session_20260908/tp_bacpipe/bacpipe_data/erirub/MNHN-SO-2020-421_extr3_0s_03_norm.wav',
 '/media/haupert/data/mes_projets/_cesco-ia/session_20260908/tp_bacpipe/bacpipe_data/erirub/MNHN-SO-2020-421_extr3_0s_04_norm.wav',
 '/media/haupert/data/mes_projets/_cesco-ia/session_20260908/tp_bacpipe/bacpipe_data/erirub/MNHN-S

---
## 3. Ground Truth Labels

Load multi-label ground truth annotations and align them to the model's timestamps.
Each row in the resulting array corresponds to the same time window as the model's
predictions, making it ready for evaluation.

Models such as BirdNET process audio in fixed-length context windows (e.g. 3
seconds). `bacpipe.ground_truth_by_model` reads `annotations.csv`, snaps the
annotations to that same time grid, and saves a `ground_truth.npy` file, so that
row *i* of the returned ground truth describes the exact same audio segment as
row *i* of the embeddings and predictions. Run it *after* the embeddings exist,
otherwise bacpipe cannot connect the embeddings to the labels.

`bacpipe.create_metadata_labels` generates a set of default labels for a
model/dataset combination, which is useful when no annotations are available.
The default labels are extracted from the filename if the format is
*<prefix>_YYYYMMDD_HHMMSS.<ext>* in order to get the *time_of_day*,
*day_of_year* and *continuous_timestamp*. The *parent directories* as well as the
*original files* are also part of the default labels. These labels are used by
the clustering and visualization steps.

By default, the results are saved in `bacpipe_results/test_data`. Two subfolders
are created: *embeddings* and *evaluations*. The *evaluations* folder is populated
with a subfolder per model (created by `ground_truth_by_model`) containing the
ground truth and the labels ready to be used by bacpipe.


In [5]:
# # Load multi-label ground truth aligned to BirdNET's 3-second time bins
# gt = bacpipe.ground_truth_by_model(
#     model='birdnet',
#     audio_dir='bacpipe/tests/test_data',
#     annotations_filename='annotations.csv',
# )
# gt

In [6]:
# # Generate metadata labels for a model/dataset combination
# dl = bacpipe.metadata_labels(
#     model='birdnet',
#     audio_dir=bacpipe.config.audio_dir
# )
# dl

---
## 4. End-to-End: `bacpipe.play`

`bacpipe.play` is the highest-level entry point. It runs the complete pipeline —
embeddings, classification, dimensionality reduction, evaluation, and an
interactive dashboard — in a single call. Ideal for a first exploration of a new
dataset across multiple models.

`play` does not return anything: all intermediate outputs are saved to disk and
the dashboard is launched automatically (unless disabled in the `config.yaml`
file). The models to run are passed with the `models` keyword (or taken from
`bacpipe.config.models`).

If a webpage with the dashboard is not automatically opened within your default
internet browser (e.g. chrome), copy then paste the URL displayed in the output
in a new page of your browser (e.g. http://localhost:5006).

The next cells select the device on which the models should run.
`bacpipe.settings.device` can be set to `'cpu'` (always works) or `'cuda'`
(only if a compatible NVIDIA GPU is available). If you do not have a GPU, leave
the default `'cpu'` to avoid CUDA errors.


In [12]:
bacpipe.config.audio_dir = str( wd_path / 'bacpipe_data') # path to directory containing audio files
bacpipe.settings.device = 'cpu' # choose 'cpu' or 'cuda' depending on your system configuration. 

bacpipe.play(
    models=['birdnet', 'birdnet_v3'],       # list of models to run. Supported models are in bacpipe.supported_models
    dim_reduction_model='umap',                          # dimensionality reduction model to use for visualization. Supported models are 'umap' and 'pca'. If 'None', no dimensionality reduction is applied.
)

Checking if the selected models require a checkpoint, and if so, if the checkpoint already exists.

birdnet checkpoint exists.

birdnet_v3 checkpoint exists.


Top end predictions for display could not be loaded. The reason could be that a previous run failed and not all predictions were saved. Regenerating the embeddings is the best chance of getting this to work. 
'NoneType' object has no attribute 'drop'


Top end predictions for display could not be loaded. The reason could be that a previous run failed and not all predictions were saved. Regenerating the embeddings is the best chance of getting this to work. 
'NoneType' object has no attribute 'drop'


Top end predictions for display could not be loaded. The reason could be that a previous run failed and not all predictions were saved. Regenerating the embeddings is the best chance of getting this to work. 
'NoneType' object has no attribute 'drop'

The port 5006 is already in use. This is most likely the case because you already 

Launching server at http://localhost:5007


Opening in existing browser session.



Top end predictions for display could not be loaded. The reason could be that a previous run failed and not all predictions were saved. Regenerating the embeddings is the best chance of getting this to work. 
'NoneType' object has no attribute 'drop'

DEBUG CLICK: {'curveNumber': 1, 'pointNumber': 1, 'pointIndex': 1, 'x': 13.163058280944824, 'y': 1.6853500604629517, 'customdata': ['erirub/MNHN-SO-2020-435_extr1_0s_00_norm.wav', 0, 3, 23, 'erirub', '{}', 1, 'birdnet']}
DEBUG CLICK: {'curveNumber': 1, 'pointNumber': 36, 'pointIndex': 36, 'x': 13.749613761901855, 'y': 1.7154940366744995, 'customdata': ['erirub/MNHN-SO-2020-423_extr3_0s_01_norm.wav', 0, 3, 10, 'erirub', '{}', 1, 'birdnet']}


In [ ]:
bacpipe.visualize_using_dashboard(
    models=['birdnet','birdnet_v3'],
    audio_dir=bacpipe.config.audio_dir,
    main_results_dir=bacpipe.settings.main_results_dir,
)


Top end predictions for display could not be loaded. The reason could be that a previous run failed and not all predictions were saved. Regenerating the embeddings is the best chance of getting this to work. 
'NoneType' object has no attribute 'drop'


Top end predictions for display could not be loaded. The reason could be that a previous run failed and not all predictions were saved. Regenerating the embeddings is the best chance of getting this to work. 
'NoneType' object has no attribute 'drop'


Top end predictions for display could not be loaded. The reason could be that a previous run failed and not all predictions were saved. Regenerating the embeddings is the best chance of getting this to work. 
'NoneType' object has no attribute 'drop'



Launching server at http://localhost:5006


Opening in existing browser session.



Top end predictions for display could not be loaded. The reason could be that a previous run failed and not all predictions were saved. Regenerating the embeddings is the best chance of getting this to work. 
'NoneType' object has no attribute 'drop'


Top end predictions for display could not be loaded. The reason could be that a previous run failed and not all predictions were saved. Regenerating the embeddings is the best chance of getting this to work. 
'NoneType' object has no attribute 'drop'


Top end predictions for display could not be loaded. The reason could be that a previous run failed and not all predictions were saved. Regenerating the embeddings is the best chance of getting this to work. 
'NoneType' object has no attribute 'drop'


Top end predictions for display could not be loaded. The reason could be that a previous run failed and not all predictions were saved. Regenerating the embeddings is the best chance of getting this to work. 
'NoneType' object has no attrib

In [ ]:
loader_obj = bacpipe.Loader(
        audio_dir=bacpipe.config.audio_dir, model_name='birdnet', use_folder_structure=True
    )
# get the embeddings as dictionnary
embeds_dict = loader_obj.embeddings(return_type="dict")

embeds_dict



finding audio files: 157it [00:00, 255135.89it/s]
Found 150 number of audio files.
No embedding files were found. Check that the path is right or if you actually have processed embeddings with birdnet_v3 for file in /media/haupert/data/mes_projets/_cesco-ia/session_20260908/tp_bacpipe/bacpipe_data.


---
## 5. Full Pipeline - Single Model

`run_pipeline_for_single_model` runs the complete bacpipe pipeline for one model:
embedding generation, classifier inference, optional dimensionality reduction,
and visualisation. It returns a `Loader` object with all results accessible and
saves all intermediate outputs to disk.

Because the outputs are stored in bacpipe's predefined folder structure,
subsequent runs with the same model/dataset combination are much faster: bacpipe
detects the already-processed embeddings and skips recomputation automatically.

It does not compute any evaluations and does not launch the dashboard. It is
intended to be integrated into existing pipelines to return embeddings and
predictions for further processing.

- `model_name`: name of the model to run. Supported models are listed in
  `bacpipe.supported_models`.
- `audio_dir`: path to the directory containing the audio files.
- `dim_reduction_model`: dimensionality reduction model to use for visualization.
  Supported models are `'umap'` and `'pca'`; pass `'None'` (the default) to skip
  dimensionality reduction entirely.


In [5]:
# All available bacpipe API endpoints
bacpipe.__all__

['play',
 'run_pipeline_for_single_model',
 'run_pipeline_for_models',
 'generate_embeddings',
 'Loader',
 'Embedder',
 'AudioHandler',
 'get_audio_files',
 'MetadataLabelMaker',
 'metadata_labels',
 'ground_truth_by_model',
 'get_dt_filename',
 'probing_pipeline',
 'run_probe_inference',
 'prepare_probe_inference',
 'clustering_pipeline',
 'run_clustering',
 'eval_clustering',
 'eval_with_silhouette',
 'benchmark',
 'model_specific_evaluation',
 'cross_model_evaluation',
 'confirm_model_name',
 'ensure_models_exist',
 'evaluation_with_settings_already_exists',
 'get_model_names',
 'make_set_paths_func',
 'visualize_using_dashboard',
 'supported_models',
 'models_needing_checkpoint',
 'TF_MODELS',
 'EMBEDDING_DIMENSIONS',
 'NEEDS_CHECKPOINT']

In [ ]:
loader_obj = bacpipe.run_pipeline_for_single_model(
    model_name='birdnet',                               # name of the model to run. Supported models are in bacpipe.supported_models
    audio_dir='bacpipe/tests/test_data',                # path to directory containing audio files  
    dim_reduction_model='umap'                          # dimensionality reduction model to use for visualization. Supported models are 'umap' and 'pca'. If 'None', no dimensionality reduction is applied. 
)
loader_obj

Checking if the selected models require a checkpoint, and if so, if the checkpoint already exists.

birdnet checkpoint exists.




###### Generating embeddings using BIRDNET ######

finding audio files: 11it [00:00, 22342.54it/s]
Found 7 number of audio files.
2026-08-26 16:48:19.382714: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-08-26 16:48:19.417449: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
cuDNN version does not match the required 9.3 for tensorflow. Device is therefore set to cpu for the tensorflow models.
Usin

---
## 6. Full Pipeline - Multiple Models

`run_pipeline_for_models` does the same as `run_pipeline_for_single_model`, but
runs the full pipeline across a list of models in one call. It returns a
dictionary of `Loader` objects keyed by model name. This makes it straightforward
to compare embeddings and predictions across models on the same dataset.

Each `Loader` exposes the same methods as above — e.g. `.metadata_dict`,
`.embeddings()` and `.predictions()`.


In [6]:
loader_dictionary = bacpipe.run_pipeline_for_models(
    models=['birdnet', 'birdnet_v3'],                  # list of models to run. Supported models are in bacpipe.supported_models
    audio_dir=bacpipe.config.audio_dir,                # path to directory containing audio files  
    dim_reduction_model='umap',                          # dimensionality reduction model to use for visualization. Supported models are 'umap' and 'pca'. If 'None', no dimensionality reduction is applied. 
    classifier_threshold= 0.1
)

display(loader_dictionary['birdnet'].metadata_dict)
display(loader_dictionary['birdnet_v3'].embeddings())

Checking if the selected models require a checkpoint, and if so, if the checkpoint already exists.

birdnet checkpoint exists.




###### Generating embeddings using BIRDNET ######

finding audio files: 157it [00:00, 161438.03it/s]
Found 150 number of audio files.
2026-09-11 16:18:49.675145: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-09-11 16:18:49.705907: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
Using device='cpu'
2026-09-11 16:18:52.344749: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed

ONNX provider: CPUExecutionProvider


Error generating embeddings for /media/haupert/data/mes_projets/_cesco-ia/session_20260908/tp_bacpipe/bacpipe_data/erirub/MNHN-SO-2020-421_extr3_0s_00_norm.wav, skipping file.
Error: index 1 is out of bounds for axis 0 with size 1
Error generating embeddings for /media/haupert/data/mes_projets/_cesco-ia/session_20260908/tp_bacpipe/bacpipe_data/erirub/MNHN-SO-2020-421_extr3_0s_01_norm.wav, skipping file.
Error: index 1 is out of bounds for axis 0 with size 1
Error generating embeddings for /media/haupert/data/mes_projets/_cesco-ia/session_20260908/tp_bacpipe/bacpipe_data/erirub/MNHN-SO-2020-421_extr3_0s_02_norm.wav, skipping file.
Error: index 1 is out of bounds for axis 0 with size 1
Error generating embeddings for /media/haupert/data/mes_projets/_cesco-ia/session_20260908/tp_bacpipe/bacpipe_data/erirub/MNHN-SO-2020-421_extr3_0s_03_norm.wav, skipping file.
Error: index 1 is out of bounds for axis 0 with size 1
Error generating embeddings for /media/haupert/data/mes_projets/_cesco-ia/se

{'model_name': 'birdnet',
 'audio_dir': '/media/haupert/data/mes_projets/_cesco-ia/session_20260908/tp_bacpipe/bacpipe_data',
 'embed_dir': '/media/haupert/data/mes_projets/_cesco-ia/session_20260908/tp_bacpipe/bacpipe_results/bacpipe_data/embeddings/2026-09-11_16-18___birdnet-bacpipe_data',
 'files': {'audio_files': ['ambient_sound/ambient_sound01.wav',
   'ambient_sound/ambient_sound02.wav',
   'erirub/MNHN-SO-2020-421_extr3_0s_00_norm.wav',
   'erirub/MNHN-SO-2020-421_extr3_0s_01_norm.wav',
   'erirub/MNHN-SO-2020-421_extr3_0s_02_norm.wav',
   'erirub/MNHN-SO-2020-421_extr3_0s_03_norm.wav',
   'erirub/MNHN-SO-2020-421_extr3_0s_04_norm.wav',
   'erirub/MNHN-SO-2020-421_extr3_0s_05_norm.wav',
   'erirub/MNHN-SO-2020-421_extr3_0s_07_norm.wav',
   'erirub/MNHN-SO-2020-423_extr3_0s_00_norm.wav',
   'erirub/MNHN-SO-2020-423_extr3_0s_01_norm.wav',
   'erirub/MNHN-SO-2020-423_extr3_0s_02_norm.wav',
   'erirub/MNHN-SO-2020-423_extr3_0s_03_norm.wav',
   'erirub/MNHN-SO-2020-423_extr3_0s_04_no

{'ambient_sound/ambient_sound01_birdnet_v3.npy': array([[0.00873414, 0.00690664, 0.00664342, ..., 0.00644558, 0.00746702,
         0.18453912]], dtype=float32),
 'ambient_sound/ambient_sound02_birdnet_v3.npy': array([[0.00679282, 0.00799298, 0.00459722, ..., 0.00504465, 0.00632125,
         0.16816929]], dtype=float32),
 'erirub/MNHN-SO-2020-421_extr3_0s_00_norm_birdnet_v3.npy': array([[0.0051894 , 0.00888881, 0.00626416, ..., 0.00598202, 0.00501374,
         0.03383933]], dtype=float32),
 'erirub/MNHN-SO-2020-421_extr3_0s_01_norm_birdnet_v3.npy': array([[0.0047637 , 0.00893007, 0.00379641, ..., 0.00491721, 0.01512499,
         0.03780156]], dtype=float32),
 'erirub/MNHN-SO-2020-421_extr3_0s_02_norm_birdnet_v3.npy': array([[0.00557942, 0.01567274, 0.00590977, ..., 0.00562894, 0.00918794,
         0.04428441]], dtype=float32),
 'erirub/MNHN-SO-2020-421_extr3_0s_03_norm_birdnet_v3.npy': array([[0.00654501, 0.01268256, 0.00695704, ..., 0.00525435, 0.00601886,
         0.0635721 ]], dtype=

---
## 7. Generating Embeddings - Full Directory (Save to Disk)

Instead of using the all-in-one pipeline functions, you can also call the `Loader`
and `Embedder` classes directly for more control. They are the core building
blocks of bacpipe, used internally by `run_pipeline_for_single_model` and
`run_pipeline_for_models`.

To process all audio files in a directory and persist the results, pass
`use_folder_structure=True` to the `Loader`. Bacpipe will create a timestamped
output directory, run inference using multithreading, and save embeddings,
metadata, and classifier predictions. Subsequent runs detect the saved files and
skip recomputation automatically.

The `loader_obj` returned by the `Loader` is the same object type returned by
`run_pipeline_for_single_model` and `run_pipeline_for_models`, so you can use it
in the same way to access embeddings, metadata, and predictions:

- `loader_obj.metadata_dict`: an overview of the audio data, where the embeddings
  were saved, per-file embedding counts, the sample rate, the segment length, and
  the total processed duration.
- `loader_obj.embeddings(return_type='array')`: all embeddings concatenated into a
  2D array where axis 0 is the time/segment axis and axis 1 the embedding
  dimension.
- `loader_obj.predictions(...)`: the pretrained classifier outputs, either as a
  per-file dictionary, a concatenated array, or a dataframe.


In [11]:
MODEL_NAME = 'birdnet'

# Create a loader object that will handle all the audio file, path and parameters needed to compute the embeddings for instance
loader_obj = bacpipe.Loader(
    audio_dir=bacpipe.config.audio_dir,
    model_name=MODEL_NAME,
    use_folder_structure=True
)

# Create an embededding object with the selected model (MODEL_NAME) passing the loader object in order to have the audio directory mapping
embed_obj = bacpipe.Embedder(
    model_name=MODEL_NAME, 
    loader=loader_obj)

# Process all files using multithreading
embed_obj.run_inference_pipeline_using_multithreading()

print('Metadata dict:')
display(loader_obj.metadata_dict)

print('Embeddings (array):')
display(loader_obj.embeddings(return_type='array'))

print('Predictions (array):')
display(loader_obj.predictions(return_type='array'))

print('Predictions (dataframe):')
# display(loader_obj.predictions(return_type='dataframe'))

Finding all generated embeddings: 150it [00:00, 266023.51it/s]
Found 150 embedding files.
finding audio files: 157it [00:00, 266493.62it/s]
Found 150 number of audio files.

### Embeddings already exist. Using embeddings in /media/haupert/data/mes_projets/_cesco-ia/session_20260908/tp_bacpipe/bacpipe_results/bacpipe_data/embeddings/2026-09-11_16-10___birdnet-bacpipe_data ###
Using device='cpu'
Skipping model.eval() because model is from tensorflow.


Metadata dict:


{'audio_dir': '/media/haupert/data/mes_projets/_cesco-ia/session_20260908/tp_bacpipe/bacpipe_data',
 'embed_dir': '/media/haupert/data/mes_projets/_cesco-ia/session_20260908/tp_bacpipe/bacpipe_results/bacpipe_data/embeddings/2026-09-11_16-10___birdnet-bacpipe_data',
 'embedding_size': 1024,
 'files': {'audio_files': ['ambient_sound/ambient_sound01.wav',
   'ambient_sound/ambient_sound02.wav',
   'erirub/MNHN-SO-2020-421_extr3_0s_00_norm.wav',
   'erirub/MNHN-SO-2020-421_extr3_0s_01_norm.wav',
   'erirub/MNHN-SO-2020-421_extr3_0s_02_norm.wav',
   'erirub/MNHN-SO-2020-421_extr3_0s_03_norm.wav',
   'erirub/MNHN-SO-2020-421_extr3_0s_04_norm.wav',
   'erirub/MNHN-SO-2020-421_extr3_0s_05_norm.wav',
   'erirub/MNHN-SO-2020-421_extr3_0s_07_norm.wav',
   'erirub/MNHN-SO-2020-423_extr3_0s_00_norm.wav',
   'erirub/MNHN-SO-2020-423_extr3_0s_01_norm.wav',
   'erirub/MNHN-SO-2020-423_extr3_0s_02_norm.wav',
   'erirub/MNHN-SO-2020-423_extr3_0s_03_norm.wav',
   'erirub/MNHN-SO-2020-423_extr3_0s_04_nor

Embeddings (array):


array([[1.2015465e+00, 0.0000000e+00, 1.8421853e-01, ..., 1.7646480e-01,
        0.0000000e+00, 2.8275006e+00],
       [4.2458090e-01, 7.9659623e-04, 3.9839163e-01, ..., 2.9775777e-01,
        0.0000000e+00, 1.8179907e+00],
       [0.0000000e+00, 1.4207257e+00, 2.1082203e-01, ..., 0.0000000e+00,
        1.6975015e+00, 0.0000000e+00],
       ...,
       [0.0000000e+00, 1.5841898e+00, 9.3311518e-02, ..., 2.8111045e+00,
        1.8780571e+00, 1.4561598e-01],
       [1.7651372e-01, 1.3401905e+00, 0.0000000e+00, ..., 2.1026316e+00,
        2.3203356e+00, 4.6246093e-02],
       [0.0000000e+00, 1.5425330e+00, 1.0109032e-01, ..., 2.9424293e+00,
        2.1064866e+00, 9.8404683e-02]], dtype=float32)

Predictions (array):


(array([], shape=(150, 0), dtype=float32), {})

Predictions (dataframe):


---
## 8. Generating Embeddings - Single File (In-Memory)

If you just want to generate embeddings for a single file without saving anything
to disk, instantiate the `Loader` and `Embedder` classes directly *without*
`use_folder_structure`. The embedding generation then runs in-memory and returns
Python objects without writing any files. This is useful for quick experimentation
or when you want to work with embeddings directly in memory.

Note that because nothing is saved to disk, `loader_obj.embeddings()` returns an
empty dict unless you keep a reference to the embedding arrays yourself (or save
them manually).


In [ ]:
# Create a Loader without folder structure — nothing is written to disk
loader_obj = bacpipe.Loader('bacpipe/tests/test_data')
embed_obj = bacpipe.Embedder('birdnet', loader_obj)

# Generate embeddings for the first audio file
embeds = embed_obj.get_embeddings_from_model(loader_obj.files[0])

print('Since embeddings were not saved, .embeddings() returns empty:')
display(loader_obj.embeddings())

print('The embeddings are still accessible via the declared variable:')
display(embeds)

print('Classifier predictions are accessible through the embedder object:')
display(embed_obj.classifier.predictions)


finding audio files: 11it [00:00, 28115.38it/s]
Found 7 number of audio files.
INFO:bacpipe:Found 7 number of audio files.
No model_name is passed, therefore no directory structure will be created.
INFO:bacpipe:No model_name is passed, therefore no directory structure will be created.
cuDNN version does not match the required 9.3 for tensorflow. Device is therefore set to cpu for the tensorflow models.
INFO:bacpipe:cuDNN version does not match the required 9.3 for tensorflow. Device is therefore set to cpu for the tensorflow models.
Using device='cuda'
INFO:bacpipe:Using device='cuda'
Skipping model.eval() because model is from tensorflow.
ERROR:bacpipe:Skipping model.eval() because model is from tensorflow.
birdnet inference took 1.73s.                                     
INFO:bacpipe:birdnet inference took 1.73s.
No embedding files were found. Check that the path is right or if you actually have processed embeddings with None for file in bacpipe/tests/test_data.


Since embeddings were not saved, .embeddings() returns empty:


None

The embeddings are still accessible via the declared variable:


array([[0.        , 0.5192538 , 0.11016142, ..., 0.02030575, 0.        ,
        0.        ],
       [0.        , 0.04109726, 0.        , ..., 0.44500944, 0.30178508,
        1.4982907 ],
       [0.        , 0.04689805, 0.        , ..., 0.        , 0.        ,
        0.35709748],
       ...,
       [0.        , 0.62402207, 0.1901902 , ..., 0.        , 0.        ,
        0.        ],
       [0.        , 0.28852096, 0.27240792, ..., 0.        , 0.        ,
        0.5682918 ],
       [0.        , 0.31145108, 0.18153723, ..., 0.        , 0.        ,
        0.45023617]], dtype=float32)

Classifier predictions are accessible through the embedder object:


tensor([[7.2337e-06, 1.2321e-06, 1.8781e-05,  ..., 9.8536e-06, 4.5944e-06,
         8.6071e-06],
        [2.6510e-06, 2.3206e-07, 8.3774e-06,  ..., 1.9020e-06, 1.8968e-06,
         4.7985e-07],
        [7.6319e-06, 1.2511e-05, 3.5127e-05,  ..., 5.9355e-05, 2.0990e-04,
         5.9185e-06],
        ...,
        [1.2915e-06, 1.7141e-07, 2.1810e-06,  ..., 1.5112e-04, 6.5344e-05,
         1.2531e-05],
        [1.5508e-05, 9.8838e-07, 7.9295e-07,  ..., 3.2521e-05, 2.3045e-05,
         1.1301e-05],
        [2.5794e-06, 3.8612e-06, 4.1147e-06,  ..., 4.6988e-05, 2.5489e-05,
         4.5167e-05]])

In [ ]:
import numpy as np
print('Get the class label of the most highest values prediction:')
label_array = np.array(embed_obj.model.classes)
class_labels = label_array[np.argmax(embed_obj.classifier.predictions, axis=1)]
display(class_labels)

Get the class label of the most highest values prediction:


array(['Eurasian Penduline Tit', 'Short-toed Treecreeper', 'Dunnock',
       'Short-toed Treecreeper', 'Short-toed Treecreeper',
       'Common Chaffinch', 'Dunnock', 'Yellow-tufted Pipit',
       'Common Chaffinch', 'Flammulated Owl', 'Common Chaffinch',
       'Dunnock', 'Short-toed Treecreeper', 'Short-toed Treecreeper',
       'Verdin', 'Common Chaffinch', 'Dunnock',
       'Eurasian Three-toed Woodpecker', 'Common Chaffinch',
       'Lesser Nighthawk', 'Common Chaffinch', 'Rock Wren'], dtype='<U34')

---
## 9. High-Level Workflow - `generate_embeddings`

`bacpipe.generate_embeddings` wraps the `Loader`/`Embedder` pattern into a single
convenient call. It runs the embedding generation pipeline for one model —
including classification using the pretrained classifier when the model ships one —
and returns a `Loader` object that exposes the metadata and the computed
embeddings. By default the results are saved in the standard folder structure;
pass `use_folder_structure=False` to only save embeddings. 
Saving cannot be completely avoided with this pipelines, as keeping
everything in memory is not always feasible. Use the 
previously explained functions if you want to avoid saving
embeddings. The embeddings can then be retrieved with 
`loader_obj.embeddings(return_type='array')`.


In [ ]:
loader_obj = bacpipe.generate_embeddings(
    model_name='birdnet',
    audio_dir='bacpipe/tests/test_data'
)

display(loader_obj.metadata_dict)
display(loader_obj.embeddings(return_type='array'))

Checking if the selected models require a checkpoint, and if so, if the checkpoint already exists.

INFO:bacpipe:Checking if the selected models require a checkpoint, and if so, if the checkpoint already exists.

birdnet checkpoint exists.

INFO:bacpipe:birdnet checkpoint exists.




###### Generating embeddings using BIRDNET ######

INFO:bacpipe:


###### Generating embeddings using BIRDNET ######

Finding all generated embeddings: 7it [00:00, 11862.68it/s]
Found 7 embedding files.
INFO:bacpipe:Found 7 embedding files.
finding audio files: 11it [00:00, 14096.35it/s]
Found 7 number of audio files.
INFO:bacpipe:Found 7 number of audio files.

### Embeddings already exist. Using embeddings in bacpipe_results/simple_use_cases/test_data/embeddings/2026-08-26_16-37___birdnet-test_data ###
INFO:bacpipe:
### Embeddings already exist. Using embeddings in bacpipe_results/simple_use_cases/test_data/embeddings/2026-08-26_16-37___birdnet-test_data ###


{'audio_dir': 'bacpipe/tests/test_data',
 'embed_dir': 'bacpipe_results/simple_use_cases/test_data/embeddings/2026-08-26_16-37___birdnet-test_data',
 'embedding_size': 1024,
 'files': {'audio_files': ['audio/FewShot/CHE_01_20190101_163410.wav',
   'audio/FewShot/CHE_02_20190101_183410.wav',
   'audio/FewShot/CHE_03_20190201_163410.wav',
   'audio/FewShot/CHE_04_20190203_175410.wav',
   'audio/UrbanSoundscape/242A2604603691DD_20250503_031300.WAV',
   'audio/UrbanSoundscape/242A2604603691DD_20250503_031400.WAV',
   'audio/UrbanSoundscape/242A2604603691DD_20250503_031500.WAV'],
  'file_lengths (s)': [63.98977083333333,
   9.890729166666667,
   8.202479166666667,
   9.351708333333333,
   30.0,
   30.0,
   30.0],
  'nr_embeds_per_file': [22, 4, 3, 4, 10, 10, 10]},
 'model_name': 'birdnet',
 'nr_embeds_total': 63,
 'sample_rate (Hz)': 48000,
 'segment_length (samples)': 144000,
 'total_dataset_length (s)': 181.4346875}

array([[0.        , 0.5192538 , 0.11016142, ..., 0.02030575, 0.        ,
        0.        ],
       [0.        , 0.04109726, 0.        , ..., 0.44500944, 0.30178508,
        1.4982907 ],
       [0.        , 0.04689805, 0.        , ..., 0.        , 0.        ,
        0.35709748],
       ...,
       [0.        , 0.36581063, 0.22898006, ..., 1.8080922 , 0.31529728,
        1.3595362 ],
       [0.        , 0.02800187, 0.02318055, ..., 0.48662075, 0.        ,
        0.610488  ],
       [0.        , 0.13184942, 0.3214704 , ..., 1.2065998 , 0.60497993,
        0.73578596]], dtype=float32)

### 9.1 Load and concatenate audio files yourself, then pass the result to bacpipe

You can also load and preprocess audio yourself and pass it directly to an
`Embedder` — useful when you need custom loading logic or want to process audio
that isn't stored on disk.

Here we load all test audio files with `librosa`, concatenate the samples into a
single long 1D numpy array, and pass it to `Embedder.generate_embeddings_from_audio_array`.
The method windows the audio into segments of the model's input length and returns
the embedding of each window. `bacpipe.ensure_models_exist` guarantees that the
model checkpoint is present locally, downloading it from the Hugging Face Hub on
first use.


In [ ]:
# Alternatively: load and concatenate audio yourself, then pass it directly to an Embedder
import librosa as lb
import numpy as np

audio_files = bacpipe.get_audio_files(
    'bacpipe/tests/test_data', return_type='str'
)

audio = []
for file in audio_files:
    aud, sr = lb.load(file)
    audio.extend(aud)
audio = np.array(audio)

# check if the model exists, if not, download it
bacpipe.ensure_models_exist(bacpipe.settings.model_base_path, ['naturebeats'])
# embed
embed_obj = bacpipe.Embedder('naturebeats')
embeds = embed_obj.generate_embeddings_from_audio_array(audio)
embeds

finding audio files: 11it [00:00, 16783.32it/s]
Found 7 number of audio files.
INFO:bacpipe:Found 7 number of audio files.
Checking if the selected models require a checkpoint, and if so, if the checkpoint already exists.

INFO:bacpipe:Checking if the selected models require a checkpoint, and if so, if the checkpoint already exists.

naturebeats checkpoint exists.

INFO:bacpipe:naturebeats checkpoint exists.

beats checkpoint exists.

INFO:bacpipe:beats checkpoint exists.

Using device='cuda'
INFO:bacpipe:Using device='cuda'
BEATs Config: {'input_patch_size': 16, 'embed_dim': 512, 'conv_bias': False, 'encoder_layers': 12, 'encoder_embed_dim': 768, 'encoder_ffn_embed_dim': 3072, 'encoder_attention_heads': 12, 'activation_fn': 'gelu', 'layer_wise_gradient_decay_ratio': 0.6, 'layer_norm_first': False, 'deep_norm': True, 'dropout': 0.0, 'attention_dropout': 0.0, 'activation_dropout': 0.0, 'encoder_layerdrop': 0.05, 'dropout_input': 0.0, 'conv_pos': 128, 'conv_pos_groups': 16, 'relative_pos

[array([-4.83361781e-02, -2.11152717e-01, -6.51871935e-02,  1.54393315e-01,
        -1.93826362e-01,  1.20965265e-01, -5.23431599e-01,  6.69796616e-02,
        -1.82797521e-01,  5.84068656e-01,  1.99373141e-02,  1.05699964e-01,
         2.06543103e-01,  7.54852295e-01, -5.18232703e-01,  2.84685850e-01,
        -3.19748342e-01, -1.65323749e-01,  5.33016026e-01, -5.15899777e-01,
        -2.02802092e-01, -3.62525135e-01, -1.18121803e-01,  1.44647360e-01,
         2.30196714e-01,  9.11206007e-02,  2.44908631e-01, -1.91879585e-01,
         7.18706027e-02,  8.88969190e-03,  2.03627851e-02, -5.12049675e-01,
        -1.74780980e-01,  8.41669068e-02, -1.79471284e-01,  1.34757712e-01,
        -5.40564209e-02, -1.13048799e-01,  1.38471007e-01,  1.67248905e-01,
         1.49838105e-01,  4.62805748e-01, -3.70644838e-01, -1.43065125e-01,
        -2.13487089e-01,  1.24424882e-02,  1.69096589e-02, -1.74494982e-01,
        -1.19828485e-01, -3.35712582e-02,  8.73064473e-02, -2.91839182e-01,
        -1.4

---
## 10. Benchmarking Classifier Performance

`bacpipe.benchmark` evaluates a model's pretrained classifier against your ground
truth annotations. It aligns predictions and ground truth to the same timestamps,
resolves label mismatches (for example hyphen or spacing differences such as
*Red-Shouldered Hawk* vs *Red Shouldered Hawk*) with a fuzzy-matching fallback,
and returns a per-species `sklearn` classification report with precision, recall,
and F1.

If predictions have already been generated for this dataset, the function loads
them from disk and runs very quickly.


In [ ]:
results = bacpipe.benchmark(
    'birdnet',
    'bacpipe/tests/test_data',
    annotations_file='annotations.csv'
)
display(results)

Fetching ground truth and mapping it to model timestamps.

INFO:bacpipe:Fetching ground truth and mapping it to model timestamps.


Multiple embeddings found for model birdnet in bacpipe_results/simple_use_cases/test_data/embeddings. Using the most recent path.

INFO:bacpipe:
Multiple embeddings found for model birdnet in bacpipe_results/simple_use_cases/test_data/embeddings. Using the most recent path.

                                                                                                  
The simultaneous labels column of the ground truth has values exceeding 1. This means you have multi-label ground truth annotations. If this should not be happening ensure the ground truth is created correcly.

The simultaneous labels column of the ground truth has values exceeding 1. This means you have multi-label ground truth annotations. If this should not be happening ensure the ground truth is created correcly.

                                                                       

{'report': {'Common Cuckoo': {'precision': 1.0,
   'recall': 0.8181818181818182,
   'f1-score': 0.9,
   'support': 11.0},
  'Eurasian Blackbird': {'precision': 1.0,
   'recall': 0.6428571428571429,
   'f1-score': 0.782608695652174,
   'support': 28.0},
  'micro avg': {'precision': 1.0,
   'recall': 0.6923076923076923,
   'f1-score': 0.8181818181818182,
   'support': 39.0},
  'macro avg': {'precision': 1.0,
   'recall': 0.7305194805194806,
   'f1-score': 0.841304347826087,
   'support': 39.0},
  'weighted avg': {'precision': 1.0,
   'recall': 0.6923076923076923,
   'f1-score': 0.8157190635451506,
   'support': 39.0},
  'samples avg': {'precision': 0.6923076923076923,
   'recall': 0.6923076923076923,
   'f1-score': 0.6923076923076923,
   'support': 39.0}},
 'gt_binary': array([[1, 0],
        [1, 0],
        [1, 0],
        [1, 0],
        [1, 0],
        [1, 0],
        [1, 0],
        [1, 0],
        [1, 0],
        [1, 0],
        [1, 0],
        [0, 1],
        [0, 1],
        [0, 1]